<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Diagnóstico de Motores con Árboles, Bosques y Redes 🌳🌲🧠
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 02
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/02_Arboles_Bosques_Redes_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 02: Árboles, Bosques Aleatorios y Redes Neuronales** ([README del módulo](../02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/README.md)) con un problema **nuevo**: diagnosticar el estado de **motores industriales** (normal, desgaste o falla inminente) a partir de lecturas de sensores.

Al terminar serás capaz de:

1. **Medir el sobreajuste** de un árbol de decisión al crecer su profundidad y elegir la profundidad con validación cruzada.
2. **Entrenar un bosque aleatorio**, usar su estimación *out-of-bag* y la **importancia por permutación** para saber qué sensores importan.
3. **Entrenar una red neuronal** (`MLPClassifier`) y reproducir su *forward pass* en **NumPy** (ReLU + *softmax*), sin librerías de aprendizaje profundo.
4. **Comparar** árboles, bosques, redes y un modelo lineal con métricas adecuadas.

> 📚 **Repasa si lo necesitas:** [Árboles de Decisión](../02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/00_Arboles_de_Decision.ipynb) · [Random Forests](../02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/01_Random_Forests.ipynb) · [Redes Neuronales](../02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/02_Redes_Neuronales.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet, sin archivos externos y sin PyTorch** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `scikit-learn`. El entrenamiento completo tarda entre uno y dos minutos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")


def generar_motores(n=2400, seed=21):
    # Estado del motor: 0 = normal, 1 = desgaste, 2 = falla inminente (según un riesgo no lineal con interacciones)
    rng = np.random.default_rng(seed)
    vib = rng.gamma(4, 0.6, n)            # vibración (mm/s)
    temp = rng.normal(70, 12, n)          # temperatura (°C)
    pres = rng.normal(5, 1.1, n)          # presión de aceite (bar)
    rpm = rng.normal(1750, 120, n)        # velocidad (rpm), sin efecto real
    corr = rng.normal(22, 4, n)           # corriente (A)
    horas = rng.uniform(0, 8000, n)       # horas de operación
    aux = rng.normal(0, 1, (n, 3))        # tres sensores auxiliares sin información
    zv, zt, zp, zc, zh = (vib - 2.4) / 1.2, (temp - 70) / 12, (pres - 5) / 1.1, (corr - 22) / 4, (horas - 4000) / 2300
    riesgo = 1.0 * zv + 0.7 * zt - 0.9 * zp + 1.0 * zc * zt + 0.8 * zv * zh + rng.normal(0, 0.45, n)
    estado = np.digitize(riesgo, np.quantile(riesgo, [0.62, 0.86]))
    return pd.DataFrame({"vibracion_mm_s": vib, "temperatura_c": temp, "presion_bar": pres, "rpm": rpm,
                         "corriente_a": corr, "horas_operacion": horas, "sensor_aux_1": aux[:, 0],
                         "sensor_aux_2": aux[:, 1], "sensor_aux_3": aux[:, 2], "estado": estado})


df = generar_motores()
X = df.drop(columns="estado")
y = df["estado"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Proporción por estado:", y.value_counts(normalize=True).sort_index().round(3).to_dict())
display(df.head())

---
## Reto 1: ¿Cuántas preguntas puede hacer un árbol antes de memorizar? 🌳

**Contexto:** un árbol sin límite de profundidad termina memorizando el entrenamiento. Vas a *medir* ese sobreajuste y elegir la profundidad con validación cruzada.

**Enunciado**
1. Para `profundidad` de 1 a 12, ajusta `DecisionTreeClassifier(max_depth=profundidad, random_state=0)` con el entrenamiento. Construye el DataFrame `curva` con tres columnas: `profundidad`, `acc_train` (exactitud en entrenamiento) y `acc_cv` (media de `cross_val_score(..., X_train, y_train, cv=cv)` usando el `cv` ya definido).
2. `mejor_prof`: la profundidad con mayor `acc_cv`.
3. `arbol`: árbol con `max_depth=mejor_prof` (y `random_state=0`) ajustado con el entrenamiento; `acc_arbol`: su exactitud en **prueba**.

**Criterios de aceptación:** `curva` tiene 12 filas; el árbol más profundo tiene un hueco entre entrenamiento y validación cruzada superior a 0.15 (sobreajuste); `mejor_prof` es coherente con `acc_cv`; `acc_arbol > 0.70`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa
curva = None        # DataFrame con columnas: profundidad, acc_train, acc_cv (12 filas)
mejor_prof = None   # profundidad con mayor acc_cv
arbol = None        # DecisionTreeClassifier(max_depth=mejor_prof, random_state=0) ajustado
acc_arbol = None    # exactitud en prueba

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert isinstance(curva, pd.DataFrame) and list(curva.columns) == ["profundidad", "acc_train", "acc_cv"] and len(curva) == 12, "'curva' debe tener 12 filas y las columnas ['profundidad', 'acc_train', 'acc_cv'] (en ese orden)."
assert list(curva["profundidad"]) == list(range(1, 13)), "La columna 'profundidad' debe recorrer 1, 2, ..., 12."
_ref = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train, cv=cv).mean() for d in range(1, 13)]
assert np.allclose(curva["acc_cv"], _ref), "acc_cv debe ser la media de cross_val_score con DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train y el 'cv' dado."
assert curva["acc_train"].iloc[-1] - curva["acc_cv"].iloc[-1] > 0.15, "El árbol de profundidad 12 debería sobreajustar (acc_train mucho mayor que acc_cv): ¿calculaste acc_train sobre el entrenamiento?"
assert curva["acc_train"].iloc[-1] > curva["acc_train"].iloc[0], "La exactitud en entrenamiento debería crecer con la profundidad."
assert mejor_prof == int(curva.loc[curva["acc_cv"].idxmax(), "profundidad"]), "mejor_prof debe ser la profundidad con mayor acc_cv."
assert isinstance(arbol, DecisionTreeClassifier) and arbol.get_params()["max_depth"] == mejor_prof and hasattr(arbol, "tree_"), "'arbol' debe ser un DecisionTreeClassifier ajustado con max_depth=mejor_prof."
assert acc_arbol is not None and abs(acc_arbol - arbol.score(X_test, y_test)) < 1e-9 and acc_arbol > 0.70, "acc_arbol debe ser la exactitud de 'arbol' en prueba y superar 0.70."
print(f"✅ Reto 1 superado: mejor profundidad = {mejor_prof} | exactitud en prueba = {acc_arbol:.3f} | hueco a profundidad 12 = {curva['acc_train'].iloc[-1] - curva['acc_cv'].iloc[-1]:.2f}")

---
## Reto 2: El bosque y sus sensores importantes 🌲

**Contexto:** promediar muchos árboles decorrelacionados reduce la varianza. Además, el bosque se **autoevalúa** con las muestras que cada árbol no vio (*out-of-bag*, OOB) y permite preguntar qué sensores importan realmente. La importancia por **permutación** mide cuánto cae el desempeño en prueba al barajar una columna.

**Enunciado**
1. `bosque = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0)` ajustado con el entrenamiento; `acc_bosque` (prueba) y `oob` (su `oob_score_`).
2. `imp_perm`: `pd.Series` (índice = columnas de `X`) con `importances_mean` de `permutation_importance(bosque, X_test, y_test, n_repeats=10, random_state=0)`.
3. `top3`: lista con los nombres de los 3 sensores más importantes, de mayor a menor.

**Criterios de aceptación:** el bosque supera al mejor árbol del Reto 1 en más de 0.03 de exactitud; OOB y prueba difieren menos de 0.06; ninguno de los 3 sensores más importantes es `rpm` ni un `sensor_aux_*` (que no tienen información).

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: completa
bosque = None      # RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0) ajustado
acc_bosque = None  # exactitud en prueba
oob = None         # estimación out-of-bag

imp_perm = None    # pd.Series de importancias por permutación (índice = X.columns)
top3 = None        # lista con los 3 nombres de sensores más importantes (de mayor a menor)

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert isinstance(bosque, RandomForestClassifier) and bosque.n_estimators == 300 and hasattr(bosque, "oob_score_"), "'bosque' debe ser un RandomForestClassifier(n_estimators=300, oob_score=True) ya ajustado."
assert acc_bosque is not None and abs(acc_bosque - bosque.score(X_test, y_test)) < 1e-9, "acc_bosque debe ser la exactitud del bosque en prueba."
assert oob is not None and abs(oob - bosque.oob_score_) < 1e-9, "oob debe ser bosque.oob_score_."
assert acc_bosque > acc_arbol + 0.03, "El bosque debería superar al mejor árbol del Reto 1 en más de 0.03 de exactitud."
assert abs(oob - acc_bosque) < 0.06, "La estimación OOB y la exactitud en prueba deberían ser parecidas (diferencia < 0.06)."
assert isinstance(imp_perm, pd.Series) and list(imp_perm.index) == list(X.columns), "imp_perm debe ser una pd.Series con índice = columnas de X."
_pi = permutation_importance(bosque, X_test, y_test, n_repeats=10, random_state=0)
assert np.allclose(imp_perm.values, _pi.importances_mean), "imp_perm debe contener importances_mean de permutation_importance(bosque, X_test, y_test, n_repeats=10, random_state=0)."
assert isinstance(top3, list) and len(top3) == 3 and top3 == list(imp_perm.sort_values(ascending=False).index[:3]), "top3 debe listar, de mayor a menor, los 3 sensores con más importancia por permutación."
assert not any(c == "rpm" or c.startswith("sensor_aux") for c in top3), "Entre los 3 más importantes no deberían aparecer 'rpm' ni los sensor_aux (no tienen información)."
print(f"✅ Reto 2 superado: bosque {acc_bosque:.3f} (árbol {acc_arbol:.3f}) | OOB = {oob:.3f} | top 3: {top3}")

---
## Reto 3: Una red neuronal y su *forward pass* a mano 🧠

**Contexto:** una red `MLPClassifier` con capas ocultas ReLU y salida *softmax* es, al final, una cadena de multiplicaciones de matrices. Para demostrar que lo entiendes, vas a **reproducir sus probabilidades con NumPy** a partir de los pesos que aprendió.

**Enunciado**
1. `mlp = Pipeline([StandardScaler, MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=0)])` ajustado con el entrenamiento (las redes **exigen** variables escaladas); `acc_mlp` es su exactitud en prueba.
2. Implementa `softmax(z)` (por filas, restando el máximo de cada fila para estabilidad numérica) y `forward(Z, coefs, intercepts)`: capas ocultas con $\mathrm{ReLU}(a) = \max(0, a)$ y capa de salida con *softmax*. Para cada capa, $A^{(l)} = \mathrm{act}\big(A^{(l-1)} W^{(l)} + \mathbf{b}^{(l)}\big)$.
3. `proba_manual = forward(Z_test, red.coefs_, red.intercepts_)`, donde `Z_test` son los datos de prueba ya escalados con el escalador **del pipeline** y `red` es su último paso.

**Criterios de aceptación:** `acc_mlp > 0.80`; tu `softmax` devuelve filas que suman 1; `proba_manual` coincide con `mlp.predict_proba(X_test)` con una tolerancia de $10^{-8}$.

**Puntaje sugerido:** 25 puntos.

In [ ]:
def relu(a):
    return np.maximum(0, a)


def softmax(z):
    # TODO: resta el máximo de cada fila, exponencia y normaliza para que cada fila sume 1
    return None


def forward(Z, coefs, intercepts):
    # TODO: recorre las capas: ocultas con relu, la última con softmax. coefs[l] es W^(l), intercepts[l] es b^(l)
    return None


# TODO: completa
mlp = None          # Pipeline([("esc", StandardScaler()), ("red", MLPClassifier(...))]) ajustado
acc_mlp = None      # exactitud en prueba
Z_test = None       # X_test escalado con el escalador del pipeline (mlp[0])
proba_manual = None # forward(Z_test, pesos y sesgos de la red entrenada)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_s = softmax(np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]]))
assert _s is not None and np.allclose(_s.sum(axis=1), 1) and np.allclose(_s[1], 1 / 3), "softmax debe devolver filas que suman 1 y ser estable con números grandes (resta el máximo de cada fila)."
assert isinstance(mlp, Pipeline) and isinstance(mlp[-1], MLPClassifier) and hasattr(mlp[-1], "coefs_"), "'mlp' debe ser un Pipeline ajustado que termina en MLPClassifier."
assert len(mlp[-1].coefs_) == 3, "Con hidden_layer_sizes=(32, 16) la red debe tener 3 matrices de pesos (2 capas ocultas + salida)."
assert acc_mlp is not None and abs(acc_mlp - mlp.score(X_test, y_test)) < 1e-9 and acc_mlp > 0.80, "acc_mlp debe ser la exactitud de 'mlp' en prueba y superar 0.80 (¿escalaste las variables?)."
assert Z_test is not None and np.allclose(Z_test, mlp[0].transform(X_test)), "Z_test debe ser X_test transformado con el escalador del pipeline (mlp[0])."
assert proba_manual is not None and proba_manual.shape == (len(X_test), 3), "proba_manual debe tener una fila por muestra de prueba y 3 columnas (una por estado)."
assert np.allclose(proba_manual, mlp.predict_proba(X_test), atol=1e-8), "Tu forward pass no reproduce mlp.predict_proba: revisa el orden W/b de cada capa, la ReLU de las ocultas y el softmax final."
print(f"✅ Reto 3 superado: exactitud de la red = {acc_mlp:.3f} y tu forward pass reproduce sus probabilidades.")

---
## Reto 4: El duelo final — ¿qué modelo despliegas? 🥊

**Contexto:** la planta debe elegir un modelo. Además de la exactitud, con clases desbalanceadas conviene mirar el $F_1$ **macro** (promedio simple del $F_1$ de cada clase).

**Enunciado**
1. Ajusta `logit = Pipeline([StandardScaler, LogisticRegression(max_iter=1000)])` con el entrenamiento (línea base lineal).
2. Construye `tabla`: `pd.DataFrame` con **índice** `["Árbol", "Bosque", "Red neuronal", "Logística"]` (en ese orden) y columnas `accuracy` y `f1_macro`, ambas en el conjunto de **prueba**, usando `arbol`, `bosque`, `mlp` y `logit`.
3. `mejor_modelo`: el nombre (índice) con mayor `f1_macro`.

**Criterios de aceptación:** las métricas se recomputan correctamente; el mejor modelo supera a la regresión logística en más de 0.05 de $F_1$ macro (hay interacciones no lineales en los datos).

**Puntaje sugerido:** 15 puntos.

In [ ]:
# TODO: completa
logit = None          # Pipeline(StandardScaler, LogisticRegression(max_iter=1000)) ajustado
tabla = None          # DataFrame: índice ["Árbol", "Bosque", "Red neuronal", "Logística"]; columnas ["accuracy", "f1_macro"]
mejor_modelo = None   # nombre del modelo con mayor f1_macro

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert isinstance(logit, Pipeline) and isinstance(logit[-1], LogisticRegression) and hasattr(logit[-1], "coef_"), "'logit' debe ser un Pipeline ajustado que termina en LogisticRegression."
assert isinstance(tabla, pd.DataFrame) and list(tabla.index) == ["Árbol", "Bosque", "Red neuronal", "Logística"], "El índice de 'tabla' debe ser ['Árbol', 'Bosque', 'Red neuronal', 'Logística'] en ese orden."
assert list(tabla.columns) == ["accuracy", "f1_macro"], "Las columnas de 'tabla' deben ser ['accuracy', 'f1_macro']."
for _n, _m in {"Árbol": arbol, "Bosque": bosque, "Red neuronal": mlp, "Logística": logit}.items():
    _p = _m.predict(X_test)
    assert abs(tabla.loc[_n, "accuracy"] - accuracy_score(y_test, _p)) < 1e-9, f"La exactitud de '{_n}' no coincide: usa accuracy_score(y_test, modelo.predict(X_test))."
    assert abs(tabla.loc[_n, "f1_macro"] - f1_score(y_test, _p, average="macro")) < 1e-9, f"El F1 macro de '{_n}' no coincide: usa f1_score(..., average='macro')."
assert mejor_modelo == tabla["f1_macro"].idxmax(), "mejor_modelo debe ser el índice de 'tabla' con mayor f1_macro."
assert tabla["f1_macro"].max() > tabla.loc["Logística", "f1_macro"] + 0.05, "El mejor modelo debería superar a la logística en más de 0.05 de F1 macro."
display(tabla.round(3))
print(f"✅ Reto 4 superado: el mejor modelo por F1 macro es «{mejor_modelo}».")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿A qué profundidad empezó a sobreajustar el árbol y cómo lo viste en `curva`?
- ¿Por qué el bosque mejoró al árbol? ¿Qué sensores resultaron importantes y qué dice eso sobre `rpm` y los `sensor_aux_*`?
- ¿Qué hizo la red neuronal que no pudo hacer el modelo lineal? Menciona el papel de las activaciones ReLU.
- Si tuvieras que desplegar un solo modelo en la planta, ¿cuál elegirías considerando métrica, interpretabilidad y costo de mantenimiento?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: curva de profundidad, sobreajuste medido y elección por validación cruzada | 20 |
| Reto 2: bosque, estimación OOB e importancia por permutación | 25 |
| Reto 3: `MLPClassifier` con escalado y *forward pass* reproducido en NumPy | 25 |
| Reto 4: tabla comparativa con métricas correctas y selección del modelo | 15 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Elegí la profundidad del árbol con validación cruzada, no con la prueba.
- [ ] Escalé las variables antes de la red neuronal y de la regresión logística.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>